# Simulated Image Data with Confounding

[Explain the dataset]

Models

[Explain the models]

Experiments

[Explain the experiments, varying N consistency, varying bz etc.]

Code Overview

[Explain functions]

---

#### Imports

In [ ]:
import os
import re
import torch
import datetime

import pandas as pd
import numpy as np

from pathlib import Path
from torch.nn import MSELoss, BCELoss
from torch.utils.data import DataLoader

from cocodeel.trainer import covar_trainer
from cocodeel.dataset import CovarDataset
from cocodeel.model import BaseNetwork, CovarNetwork
from cocodeel.posthoc_model import PostHocCovarNetwork
from cocodeel.benchmarking.posthoc_model import PostHocOrthNetwork, SemiStructuredNetwork

from experiments.simulation_images.backbone import TrafficBackbone
from experiments.simulation_images.utils import simulate_dataloader, simulate_dataloaders_split, simulate_traffic_light_data, evaluate_model

print(torch.__version__)
print(torch.version.cuda)

#### Default Parameters

In [ ]:
simulation_params = {
    'n': 25600, 'bz': 1., 'b2': 1., 'b3': 1.,
    'cv1': 0.8, 'cv2': 0.5, 'sdy': 1.
}
trainer_params = {
    'device': 'cuda:1', 'loss_fn': MSELoss(), 'epochs': 1000, 'lr': 1e-3, 'weight_decay': 1e-4, 'patience': 12
}
model_params = {
    'backbone': TrafficBackbone,
    'backbone_params': {"out_features": 32},
    'num_covariates': 1
}
posthoc_configs = {
    "posthoc": dict(cls=PostHocCovarNetwork),
    "posthoc_lam0": dict(cls=PostHocCovarNetwork, fit_kwargs={"lam": 0.0}),
    "posthoc_orth": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True}),
    "posthoc_orth_lam0": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True}, fit_kwargs={"lam": 0.0}),
    "posthoc_web": dict(cls=PostHocOrthNetwork)
}

#### Functions for running the experiments

* simulate_and_fit

In [ ]:
def simulate_and_fit(simulation_params, posthoc_configs, model_params, trainer_params, seed=0, covar_model=False):
    """Simulate once, then fit a DNN-only baseline on the full draw and a
    split-recipe refit on disjoint halves.

    - `base_full` trains on the full N (DNN-only reference, unchanged from paper).
    - `base_half` trains on half_A; posthoc refits on the disjoint half_B.

    The refit uses the split recipe (Pagan 1984 generated regressors): on
    half_B, H = phi(X; theta*) is a deterministic function of X, so FWL+ridge
    is unbiased. Same-sample refits are no longer produced.
    """
    torch.manual_seed(seed)

    full, half_A, half_B = simulate_dataloaders_split(simulation_params, seed=seed)
    full_tr, full_va = full
    hA_tr, hA_va = half_A
    hB_tr, hB_va = half_B

    # --- DNN-only baseline on full N ---
    base_full = covar_trainer(
        model=BaseNetwork,
        model_params=model_params,
        train_loader=full_tr,
        val_loader=full_va,
        **trainer_params
    ).center_effects(full_tr)

    # --- Split-recipe backbone on N/2 ---
    base_half = covar_trainer(
        model=BaseNetwork,
        model_params=model_params,
        train_loader=hA_tr,
        val_loader=hA_va,
        **trainer_params
    ).center_effects(hA_tr)

    # --- Covariate (end-to-end) model on full N, if requested ---
    if covar_model:
        covar_model_obj = covar_trainer(
            model=CovarNetwork,
            model_params=model_params,
            train_loader=full_tr,
            val_loader=full_va,
            **trainer_params
        ).center_effects(full_tr)

    # --- Posthoc refits: SPLIT recipe only ---
    posthoc_models = {}
    for name, cfg in posthoc_configs.items():
        init_kwargs = cfg.get("init_kwargs", {})
        fit_kwargs = cfg.get("fit_kwargs", {})
        model_cls = cfg["cls"]

        m_split = model_cls(base_half, num_covariates=model_params["num_covariates"], **init_kwargs)
        posthoc_models[name] = m_split.fit(hB_tr, hB_va, **fit_kwargs) if hasattr(m_split, "fit") else m_split

    # --- Semi-Structured Network on full (covar + orth) ---
    if covar_model:
        ssn_model = SemiStructuredNetwork(covar_model_obj)
        posthoc_models["ssn"] = ssn_model.fit(full_tr)

    # --- Package ---
    result = {
        "seed": seed,
        "simulation_params": simulation_params,
        "model_params": model_params,
        "trainer_params": trainer_params,
        "models": {"base_full": base_full, "base_half": base_half, **posthoc_models},
    }
    if covar_model:
        result["models"]["covar"] = covar_model_obj
    return result


In [ ]:
def run_experiment(name, n_runs, simulation_params, posthoc_configs, model_params, trainer_params, covar_model=False):
    start_time = datetime.datetime.now()

    # Create descriptive experiment name
    n = simulation_params["n"]
    bz = simulation_params["bz"]
    cv1 = simulation_params["cv1"]
    p = model_params['num_covariates']
    q = model_params["backbone_params"]["out_features"]
    timestamp = start_time.strftime("%Y%m%d-%H%M%S")

    experiment_name = (
        f"{name}/n={n}_bz={bz}_cv1={cv1}_p={p}_q={q}_{timestamp}"
    )

    print(f" Starting experiment: {experiment_name}")
    print(f" Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Setup output directory
    out_dir = Path(f"outputs/simulation_images") / experiment_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Run experiment for multiple seeds
    for run_id in range(n_runs):
        print(f" Run {run_id + 1}/{n_runs} started at {datetime.datetime.now():%H:%M:%S}")

        result_dict = simulate_and_fit(
            simulation_params, posthoc_configs, model_params, trainer_params, seed=run_id, covar_model=covar_model
        )

        # Save model weights
        for model_name, model in result_dict["models"].items():
            model_path = out_dir / f"{model_name}_{run_id}.pth"
            torch.save(model.state_dict(), model_path)

        print(f" Saved models for run {run_id} to {out_dir}")

    print(f" Experiment '{experiment_name}' completed in {datetime.datetime.now() - start_time}")
    return experiment_name

---
## Experiments
#### Binary Data - Increasing $\beta_Z$

In [ ]:
simulation_params = {
    'n': 25600, 'bz': 1., 'b2': 1., 'b3': 1.,
    'cv1': 0.8, 'cv2': 0.5, 'sdy': 1., 'outcome_type': 'binary'
}
trainer_params = {
    'device': 'cuda:1', 'loss_fn': BCELoss(), 'epochs': 1000, 'lr': 1e-3, 'weight_decay': 1e-4, 'patience': 12
}
model_params = {
    'backbone': TrafficBackbone,
    'backbone_params': {"out_features": 32},
    'num_covariates': 1,
    'link': 'logit'
}
posthoc_configs = {
    "posthoc": dict(cls=PostHocCovarNetwork,  fit_kwargs={"max_iters": 25}),
    "posthoc_orth": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True}, fit_kwargs={"max_iters": 25}),
    "posthoc_web": dict(cls=PostHocOrthNetwork)
}

In [ ]:
# Run experiment for different values of bz (over different n).
for bz in [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4]:
    for n in [200, 400, 800, 1600, 3200, 6400, 12800, 25600]:
        experiment_name = run_experiment(
            name="binary_increasing_bz",
            n_runs=100,
            simulation_params={**simulation_params, 'bz': bz, 'n': n},
            posthoc_configs=posthoc_configs,
            model_params=model_params,
            trainer_params=trainer_params
        )

#### Increasing $\beta_Z$

In [ ]:
simulation_params = {
    'n': 25600, 'bz': 1., 'b2': 1., 'b3': 1.,
    'cv1': 0.8, 'cv2': 0.5, 'sdy': 1., 'outcome_type': 'continuous'
}
trainer_params = {
    'device': 'cuda:1', 'loss_fn': MSELoss(), 'epochs': 1000, 'lr': 1e-3, 'weight_decay': 1e-4, 'patience': 12
}
model_params = {
    'backbone': TrafficBackbone,
    'backbone_params': {"out_features": 32},
    'num_covariates': 1
}
posthoc_configs = {
    "posthoc": dict(cls=PostHocCovarNetwork),
    "posthoc_lam0": dict(cls=PostHocCovarNetwork, fit_kwargs={"lam": 0.0}),
    "posthoc_orth": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True}),
    "posthoc_orth_lam0": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True}, fit_kwargs={"lam": 0.0}),
    "posthoc_web": dict(cls=PostHocOrthNetwork)
}

In [ ]:
# Run experiment for different values of bz (over different n).
for n in [200, 400, 800, 1600, 3200, 6400, 12800, 25600]:
    for bz in [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4]:
        experiment_name = run_experiment(
            name="increasing_bz",
            n_runs=100,
            simulation_params={**simulation_params, 'bz': bz, 'n': n},
            posthoc_configs=posthoc_configs,
            model_params=model_params,
            trainer_params=trainer_params
        )

#### Increasing $corr(X,Z)$


In [ ]:
simulation_params = {
    'n': 25600, 'bz': 1., 'b2': 1., 'b3': 1.,
    'cv1': 0.8, 'cv2': 0.5, 'sdy': 1., 'outcome_type': 'continuous'
}
trainer_params = {
    'device': 'cuda:1', 'loss_fn': MSELoss(), 'epochs': 1000, 'lr': 1e-3, 'weight_decay': 1e-4, 'patience': 12
}
model_params = {
    'backbone': TrafficBackbone,
    'backbone_params': {"out_features": 32},
    'num_covariates': 1,
    'link': 'identity'
}
posthoc_configs = {
    "posthoc": dict(cls=PostHocCovarNetwork),
    "posthoc_orth": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True})
}

In [ ]:
for n in [200, 400, 800, 1600, 3200, 6400, 12800, 25600]:
    for cv in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        experiment_name = run_experiment(
            name="increasing_cv",
            n_runs=100,
            simulation_params={**simulation_params, 'cv1': cv, 'n': n},
            posthoc_configs=posthoc_configs,
            model_params=model_params,
            trainer_params=trainer_params
        )

#### Increasing $q$

In [ ]:
for n in [200, 400, 800, 1600, 3200, 6400, 12800, 25600]:
    for q in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
        experiment_name = run_experiment(
            name="increasing_q",
            n_runs=100,
            simulation_params={**simulation_params, 'n': n},
            posthoc_configs=posthoc_configs,
            model_params={**model_params, "backbone_params": {"out_features": q}},
            trainer_params=trainer_params
        )

#### Increasing $p$ (todo!)

In [ ]:
simulation_params = {
    'n': 25600, 'bz': 1., 'b2': 1., 'b3': 1.,
    'cv1': 0.8, 'cv2': 0.5, 'sdy': 1., 'outcome_type': 'continuous'
}
trainer_params = {
    'device': 'cuda:1', 'loss_fn': MSELoss(), 'epochs': 1000, 'lr': 1e-3, 'weight_decay': 1e-4, 'patience': 12
}
model_params = {
    'backbone': TrafficBackbone,
    'backbone_params': {"out_features": 32},
    'num_covariates': 1,
    'link': 'identity'
}
posthoc_configs = {
    "posthoc": dict(cls=PostHocCovarNetwork),
    "posthoc_orth": dict(cls=PostHocCovarNetwork, init_kwargs={"orthogonalize": True})
}

In [ ]:
for n in [200, 400, 800, 1600, 3200, 6400, 12800, 25600]:
    for p in [1, 2, 4, 8, 16]:
        experiment_name = run_experiment(
            name="increasing_p",
            n_runs=100,
            simulation_params={**simulation_params, 'n': n, 'n_covars': p},
            posthoc_configs=posthoc_configs,
            model_params={**model_params, 'num_covariates': p},
            trainer_params=trainer_params
        )

---
## Experiment: Concurvity

In [ ]:
for n in [200, 400, 800, 1600, 3200, 6400, 12800, 25600]:
    experiment_name = run_experiment(
        name="concurvity",
        n_runs=100,
        simulation_params={**simulation_params, 'bz': 1.5, 'n': n},
        posthoc_configs=posthoc_configs,
        model_params=model_params,
        trainer_params=trainer_params,
        covar_model=True
    )

---

## Laboratory (discard this)

In [ ]:
n = 800
n_covars = 16
h=20
w=60
circle_radius=8
bz=1.
b2=1.
b3=1.
cv1=0.8
cv2=0.5
sdy=1.
seed=0
outcome_type='continuous'

In [ ]:
# 1. Covariate Z
Z = torch.rand(n, n_covars)  # uniform(0,1)

# 2. Latent variables
v1_raw = torch.rand(n, 1)
v2_raw = torch.rand(n, 1)
v3 = torch.rand(n, 1) # independent

# Correlate v1 and v2 with a (linear) function of Z.
Zcorr = Z.mean(dim=1, keepdim=True)  # average if multiple covariates
v1 = (1 - cv1) * v1_raw + cv1 * Zcorr
v2 = (1 - cv2) * v2_raw + cv2 * Zcorr

# 3. Build X images.
X = torch.zeros((n, 1, h, w))

centers = [(h//2, w//6), (h//2, w//2), (h//2, 5*w//6)]

def circle_mask(h, w, center, radius):
    Y, Xg = torch.meshgrid(torch.arange(h), torch.arange(w), indexing='ij')
    dist = (Xg - center[1])**2 + (Y - center[0])**2
    return dist <= radius**2

mask1 = circle_mask(h, w, centers[0], circle_radius)
mask2 = circle_mask(h, w, centers[1], circle_radius)
mask3 = circle_mask(h, w, centers[2], circle_radius)

for i in range(n):
    X[i, 0][mask1] = v1[i]
    X[i, 0][mask2] = v2[i]
    X[i, 0][mask3] = v3[i]

plt.imshow(X[0,:].squeeze(), vmin=0, vmax=1)

In [ ]:
# 1. Covariate Z
Z = torch.rand(n, n_covars)  # uniform(0,1)

# 2. Latent variables
v1_raw = torch.rand(n, 1)
v2_raw = torch.rand(n, 1)
v3 = torch.rand(n, 1)  # independent


# ---------- helper: split a mask into vertical strips ----------
def split_mask_into_vertical_strips(mask, n_strips):
    """
    Split a boolean mask into n_strips vertical strips.
    Returns a list of boolean masks.
    """
    ys, xs = mask.nonzero(as_tuple=True)
    x_min, x_max = xs.min(), xs.max() + 1
    width = x_max - x_min

    base = width // n_strips
    remainder = width % n_strips

    strips = []
    start = x_min
    for j in range(n_strips):
        wj = base + (1 if j < remainder else 0)
        end = start + wj

        strip = torch.zeros_like(mask)
        strip[ys, xs] = (xs >= start) & (xs < end)
        strips.append(strip)

        start = end

    return strips


# 3. Build X images.
X = torch.zeros((n, 1, h, w))

centers = [(h // 2, w // 6), (h // 2, w // 2), (h // 2, 5 * w // 6)]

def circle_mask(h, w, center, radius):
    Y, Xg = torch.meshgrid(
        torch.arange(h), torch.arange(w), indexing='ij'
    )
    dist = (Xg - center[1])**2 + (Y - center[0])**2
    return dist <= radius**2


mask1 = circle_mask(h, w, centers[0], circle_radius)
mask2 = circle_mask(h, w, centers[1], circle_radius)
mask3 = circle_mask(h, w, centers[2], circle_radius)

# Split confounded circles into strips
mask1_strips = split_mask_into_vertical_strips(mask1, n_covars)
mask2_strips = split_mask_into_vertical_strips(mask2, n_covars)

# Fill images
for i in range(n):

    # v1 circle (confounded, split by Z)
    for j, strip in enumerate(mask1_strips):
        v1_ij = (1 - cv1) * v1_raw[i] + cv1 * Z[i, j]
        X[i, 0][strip] = v1_ij

    # v2 circle (confounded, split by Z)
    for j, strip in enumerate(mask2_strips):
        v2_ij = (1 - cv2) * v2_raw[i] + cv2 * Z[i, j]
        X[i, 0][strip] = v2_ij

    # v3 circle (unconfounded)
    X[i, 0][mask3] = v3[i]


# Visualization
plt.imshow(X[0, 0], vmin=0, vmax=1)
plt.colorbar()


In [ ]:
from experiments.simulation_images.dataset import circle_mask, split_mask_into_vertical_strips

import matplotlib.pyplot as plt

img = [] # some array of images
frames = [] # for storing the generated images


h, w = 20, 60
centers = [(h // 2, w // 6), (h // 2, w // 2), (h // 2, 5 * w // 6)]

circle_mask(h, w, centers[0], 8)

ms = split_mask_into_vertical_strips(circle_mask(h, w, centers[0], 8), n_strips=16)

fig = plt.figure()
for i, m in enumerate(ms):
    plt.subplot(4, 4, i + 1)
    plt.imshow(m, cmap='gray')
    plt.axis('off')
plt.show()


In [ ]:
# Manual testing.

# Data for training.
# Simulate data.
X, Z, y, v1, v2, v3, fx, fz, fr = experiments.traffic_light.dataset.simulate_traffic_light_data(n=400, seed=1)
N = X.shape[0]
# Create training and validation.
train_data = CovarDataset(X[:N // 2], Z[:N // 2], y[:N // 2])
val_data = CovarDataset(X[N // 2:], Z[N // 2:], y[N // 2:])
# Create dataloaders.
train_loader = DataLoader(train_data, batch_size= N // 30, shuffle=True)
val_loader = DataLoader(val_data, batch_size= N // 30, shuffle=False)

# Load base model.
model = BaseNetwork(backbone=experiments.traffic_light.backbone.TrafficBackbone, backbone_params={"out_features": 32})
model.load_state_dict(torch.load(
    f"../../outputs/traffic_light/consistency_bz=1.0_corrv1=0.8/traffic_light_nobs=400_nruns=100_20251001-174205/base_1.pth"))
model.eval()

# Load posthoc model.
posthoc_model = PostHocCovarNetwork(model, num_covariates=Z.shape[1])
posthoc_model.load_state_dict(torch.load(
    f"../../outputs/traffic_light/consistency_bz=1.0_corrv1=0.8/traffic_light_nobs=400_nruns=100_20251001-174205/posthoc_1.pth"))
posthoc_model.eval()

In [ ]:
device = 'cpu'
features = []
covariates = []
targets = []

# Data for training.
# Simulate data.
X, Z, y, v1, v2, v3, fx, fz, fr = experiments.traffic_light.dataset.simulate_traffic_light_data(n=400, seed=1)
N = X.shape[0]
# Create training and validation.
train_data = CovarDataset(X[:N // 2], Z[:N // 2], y[:N // 2])
val_data = CovarDataset(X[N // 2:], Z[N // 2:], y[N // 2:])
# Create dataloaders.
train_loader = DataLoader(train_data, batch_size= N // 30, shuffle=True)
val_loader = DataLoader(val_data, batch_size= N // 30, shuffle=False)

with torch.no_grad():
    for batch in train_loader:
        x = batch["X"].to(device)
        z = batch["Z"].to(device)
        y = batch["y"].to(device)

        x_feat = posthoc_model.backbone(x)
        x_feat = posthoc_model.center_x(x_feat)
        z = posthoc_model.center_z(z)

        features.append(x_feat)
        covariates.append(z)
        targets.append(y)

    for batch in val_loader:
        x = batch["X"].to(device)
        z = batch["Z"].to(device)
        y = batch["y"].to(device)

        x_feat = posthoc_model.backbone(x)
        x_feat = posthoc_model.center_x(x_feat)
        z = posthoc_model.center_z(z)

        features.append(x_feat)
        covariates.append(z)
        targets.append(y)

F = torch.cat(features, dim=0)
P = torch.cat(covariates, dim=0)
o = torch.cat(targets, dim=0)
D = torch.cat([F, P], dim=1)

In [ ]:
print("Base model")
print(model.fx.weight)
print("Posthoc model")
posthoc_model.fit(dataloader=train_loader, lam=torch.tensor([0.0, 0.0]))
print(posthoc_model.fx.weight)
print(posthoc_model.fz.weight)
print(posthoc_model.lam)

In [ ]:
F_R = pd.DataFrame(F.numpy())
F_R.columns = [f"X{i+1}" for i in range(F.shape[1])]
P_R = pd.DataFrame(P.numpy())
P_R.columns = [f"Z{i+1}" for i in range(P.shape[1])]
o_R = pd.DataFrame(o.numpy())
o_R.columns = ["y"]
data = pd.concat([F_R, P_R, o_R], axis=1)
# save to csv for R


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R -i data -w 8 -h 4 --units in -r 200

library(glmnet)
X = as.matrix(data[,1:33])
y = as.matrix(data[,34])
p.fac = rep(1,34)
p.fac[34] = 0  # No penalty on covariate.

fit <- glmnet(X, y, alpha=0, penalty_factor = p.fac)
plot(fit, label=TRUE)

# Using glmnet function to build the ridge regression in r
ridge_cv <- cv.glmnet(X, y, alpha=0, penalty_factor = p.fac)
plot(ridge_cv)

best_lambda <- ridge_cv$lambda.min
print(best_lambda)

# Fit using best lambda.
m <- glmnet(X, y, alpha=0, penalty_factor = p.fac, lambda  = best_lambda)
coef(m)

In [ ]:
# Compute predictions on validation set
device = 'cpu'

X, Z, y, v1, v2, v3, fx, fz, fr = experiments.traffic_light.dataset.simulate_traffic_light_data(n=800, seed=1234)
N = X.shape[0]
test_data = CovarDataset(X, Z, y)
test_loader = DataLoader(test_data, batch_size= N // 30, shuffle=False)

val_losses = []

for batch in test_loader:
    X_val, Z_val, y_val = batch["X"].to(device), batch["Z"].to(device), batch["y"].to(device)
    # Forward pass (linear model: yhat = fx*X + fz*Z + intercept)
    yhat = model(X_val).squeeze().detach().cpu()
    loss = ((y_val - yhat)**2).mean()
    val_losses.append(loss.item())

mean_val_loss = sum(val_losses) / len(val_losses)
print(mean_val_loss)

In [ ]:
print(mean_val_loss)  # 1.57676213234663 (lam = 1)

In [ ]:
# Suppose val_loader yields batches: (X_val, Z_val, y_val)
# Candidate lambda values
lambda_grid = torch.logspace(-5, 5, 20)  # 20 values from 1e-5 to 1e2

best_lam = None
best_loss = float('inf')

for lam in lambda_grid:
    # Fit ridge on training data (replace X_train, Z_train, y_train with your tensors)
    posthoc_model._linear_fit_from_loader(train_loader, lam=lam.item())

    # Compute predictions on validation set
    val_losses = []
    for batch in val_loader:
        X_val, Z_val, y_val = batch["X"].to(device), batch["Z"].to(device), batch["y"].to(device)
        # Forward pass (linear model: yhat = fx*X + fz*Z + intercept)
        yhat = posthoc_model(X, Z).squeeze().detach().cpu()
        loss = ((y_val - yhat)**2).mean()
        val_losses.append(loss.item())

    mean_val_loss = sum(val_losses) / len(val_losses)

    print(f"Lambda: {lam.item():.5f}, validation MSE: {mean_val_loss:.5f}")

    if mean_val_loss < best_loss:
        best_loss = mean_val_loss
        best_lam = lam.item()

print(f"Optimal lambda: {best_lam:.5f}, validation MSE: {best_loss:.5f}")